# pybioclip

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imageomics/evolution-workshop-2026/blob/main/docs/tutorials/notebooks/pybioclip.ipynb)

> Classify focal species and generate image embeddings programmatically using BioCLIP models, without complex ML infrastructure.

Part of the [Designing for Discovery](https://imageomics.github.io/evolution-workshop-2026/) workshop at [Evolution 2026](https://www.evolutionmeeting.org/) in Cleveland, OH.

**To run this notebook:** click the *Open In Colab* badge above (no local setup required), then run the cells top to bottom (use Runtime, then Run all). A GPU runtime is optional but speeds things up (use Runtime, then Change runtime type, then GPU).

## Learning objectives

By the end of this tutorial, you will be able to:

1. Install `pybioclip` and run it on example images.
2. Classify an organism against the full Tree of Life and interpret the ranked predictions.
3. Score an image against your own set of custom labels.
4. Generate image embeddings (feature vectors) for downstream tasks such as similarity search and clustering.

## Prerequisites

- **Python:** >= 3.10 (Colab satisfies this by default).
- **Packages:** `pybioclip` (installed below). Pulls in `torch`, `torchvision`, `open_clip_torch`.
- **Data:** two example images downloaded in the Setup step, so no data prep is required.
- **Prior knowledge:** basic Python. No machine-learning background needed.

## Background

[BioCLIP](https://imageomics.github.io/bioclip/) is a vision foundation model trained on the [TreeOfLife](https://huggingface.co/datasets/imageomics/TreeOfLife-10M) dataset to understand images of organisms across the tree of life. [`pybioclip`](https://imageomics.github.io/pybioclip/) is a small Python library (and CLI) that wraps BioCLIP so you can classify images and extract embeddings in a few lines, with no model-loading or tensor wrangling required.

Two ideas you'll use below:

- **Classification** turns an image into a ranked list of labels with scores. Against the Tree of Life you get taxonomic predictions from kingdom down to species; with custom labels you get scores for terms *you* supply.
- **Embeddings** turn an image into a fixed-length vector that captures its visual content. Similar organisms land near each other in this space, which is the basis for similarity search, clustering, and downstream classifiers.

## Setup

Install `pybioclip` and download two example images (a brown bear and a domestic cat) from the BioCLIP demo.

In [ ]:
%pip install -q pybioclip

In [ ]:
import urllib.request

EXAMPLES = {
    "Ursus-arctos.jpeg": "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Ursus-arctos.jpeg",
    "Felis-catus.jpeg": "https://huggingface.co/spaces/imageomics/bioclip-demo/resolve/main/examples/Felis-catus.jpeg",
}
for name, url in EXAMPLES.items():
    urllib.request.urlretrieve(url, name)
    print("downloaded", name)

In [ ]:
from PIL import Image

Image.open("Ursus-arctos.jpeg")

## Step 1: Classify against the Tree of Life

`TreeOfLifeClassifier` ranks an image against the full BioCLIP taxonomy. Ask for predictions at a given `Rank` (e.g. `Rank.SPECIES`); each result is a dict with the taxonomic fields plus a `score`.

In [ ]:
from bioclip import TreeOfLifeClassifier, Rank

classifier = TreeOfLifeClassifier()
predictions = classifier.predict("Ursus-arctos.jpeg", Rank.SPECIES)

for p in predictions:
    print(f"{p['species']:30s} ({p['common_name']:20s})  {p['score']:.4f}")

Try a coarser rank, or raise `k` to see more candidates:

In [ ]:
classifier.predict("Ursus-arctos.jpeg", Rank.GENUS, k=3)

## Step 2: Classify with your own labels

When you only care about a specific set of categories, `CustomLabelsClassifier` scores an image against the labels you provide. Each result is a dict with `classification` and `score`.

In [ ]:
from bioclip import CustomLabelsClassifier

classifier = CustomLabelsClassifier(["bear", "cat", "fish", "bird"])
for p in classifier.predict("Felis-catus.jpeg"):
    print(f"{p['classification']:8s}  {p['score']:.4f}")

## Step 3: Generate image embeddings

`create_image_features` returns a normalized feature vector per image as a `torch.Tensor`. These embeddings are the foundation for similarity search, clustering, and training lightweight downstream classifiers.

In [ ]:
from bioclip import TreeOfLifeClassifier

classifier = TreeOfLifeClassifier()
features = classifier.create_image_features(["Ursus-arctos.jpeg", "Felis-catus.jpeg"])
print("features shape:", tuple(features.shape))  # (num_images, embedding_dim)

Because the vectors are L2-normalized, the dot product between two embeddings is their cosine similarity, a quick check of how visually alike two organisms are:

In [ ]:
bear, cat = features[0], features[1]
similarity = (bear @ cat).item()
print(f"cosine similarity (bear vs. cat): {similarity:.4f}")

## Your turn

_TBD: hands-on prompts for participants. Ideas:_

- _Upload your own image (`Files` panel or `google.colab.files.upload()`) and classify it._
- _Compare predictions at different ranks, or with custom vs. Tree of Life labels._
- _Embed a small folder of images and rank them by similarity to a query image._

## Summary

You installed `pybioclip` and used BioCLIP to (1) classify an image against the Tree of Life, (2) score it against custom labels, and (3) generate image embeddings and measure similarity, all without managing ML infrastructure.

**Next steps**

- [pybioclip documentation](https://imageomics.github.io/pybioclip/): full Python API and CLI reference.
- [Command-line tutorial](https://imageomics.github.io/pybioclip/command-line-tutorial/): `bioclip predict` and `bioclip embed`.
- Other workshop tutorials at [Designing for Discovery](https://imageomics.github.io/evolution-workshop-2026/).

## Troubleshooting

| Problem | Solution |
|---------|----------|
| First prediction is slow | The model weights download once on first use, then are cached for the session. |
| Out-of-memory or very slow | Switch to a GPU runtime using Runtime, then Change runtime type, then GPU. |
| Example image download fails | Re-run the Setup cell, or upload your own image via the Colab `Files` panel. |